In [60]:
import pandas as pd
import sqlite3
import os

db_file = "test_data_edited.db"
conn = sqlite3.connect(os.path.join(os.getcwd(), "notebooks", "example", db_file))

In [61]:
df = pd.read_csv(os.path.join(os.getcwd(), "notebooks", "example", "test_data_edited.tsv"), sep="\t", index_col="ID")
df

,Gene Symbols,Synonyms,Aliases
ID,,,
BLA:1,A,"A1,A2,A3","aa,ab,ac"
BLA:2,B,"B1,B2","ba,bb"
BLA:3,C,"C1,C2,C3,C4","ca,cb,cc"


In [62]:
df.index = df.index.str.split(":").str[1].astype(int)
df

,Gene Symbols,Synonyms,Aliases
ID,,,
1,A,"A1,A2,A3","aa,ab,ac"
2,B,"B1,B2","ba,bb"
3,C,"C1,C2,C3,C4","ca,cb,cc"


In [63]:
df.index.rename("id", inplace=True)
df

,Gene Symbols,Synonyms,Aliases
id,,,
1,A,"A1,A2,A3","aa,ab,ac"
2,B,"B1,B2","ba,bb"
3,C,"C1,C2,C3,C4","ca,cb,cc"


In [64]:
df.columns = df.columns.str.lower()
df

,gene symbols,synonyms,aliases
id,,,
1,A,"A1,A2,A3","aa,ab,ac"
2,B,"B1,B2","ba,bb"
3,C,"C1,C2,C3,C4","ca,cb,cc"


In [65]:
df.rename(columns={"gene symbols": "gene_symbols"}, inplace=True)
df

,gene_symbols,synonyms,aliases
id,,,
1,A,"A1,A2,A3","aa,ab,ac"
2,B,"B1,B2","ba,bb"
3,C,"C1,C2,C3,C4","ca,cb,cc"


In [66]:
df["synonym"] = df["synonyms"].str.split(",")
df

,gene_symbols,synonyms,aliases,synonym
id,,,,
1,A,"A1,A2,A3","aa,ab,ac","[A1, A2, A3]"
2,B,"B1,B2","ba,bb","[B1, B2]"
3,C,"C1,C2,C3,C4","ca,cb,cc","[C1, C2, C3, C4]"


In [67]:
df_alias = df.synonym.explode()
df_alias

id
1    A1
1    A2
1    A3
2    B1
2    B2
3    C1
3    C2
3    C3
3    C4
Name: synonym, dtype: str

In [68]:
df_alias = df_alias.to_frame()
df_alias

,synonym
id,
1,A1
1,A2
1,A3
2,B1
2,B2
3,C1
3,C2
3,C3
3,C4


In [69]:
df_alias["gene_id"] = df_alias.index
df_alias

,synonym,gene_id
id,,
1,A1,1
1,A2,1
1,A3,1
2,B1,2
2,B2,2
3,C1,3
3,C2,3
3,C3,3
3,C4,3


In [70]:
df_alias.reset_index(drop=True, inplace=True)
df_alias

,synonym,gene_id
0,A1,1
1,A2,1
2,A3,1
3,B1,2
4,B2,2
5,C1,3
6,C2,3
7,C3,3
8,C4,3


In [71]:
df_alias.index += 1
df_alias

,synonym,gene_id
1,A1,1
2,A2,1
3,A3,1
4,B1,2
5,B2,2
6,C1,3
7,C2,3
8,C3,3
9,C4,3


In [72]:
df_alias.index.rename("id", inplace=True)
df_alias

,synonym,gene_id
id,,
1,A1,1
2,A2,1
3,A3,1
4,B1,2
5,B2,2
6,C1,3
7,C2,3
8,C3,3
9,C4,3


In [73]:
df_alias.to_sql("synonym", conn, index_label="id", if_exists="replace")

9

In [74]:
df_gene = df[["gene_symbols"]].copy()
df_gene.rename(columns={"gene_symbols": "symbol"}, inplace=True)
df_gene


,symbol
id,
1,A
2,B
3,C


In [75]:
df_gene.to_sql("gene", conn, index_label="id", if_exists="replace")

3

In [76]:
df["alias"] = df["aliases"].str.split(",")
df

,gene_symbols,synonyms,aliases,synonym,alias
id,,,,,
1,A,"A1,A2,A3","aa,ab,ac","[A1, A2, A3]","[aa, ab, ac]"
2,B,"B1,B2","ba,bb","[B1, B2]","[ba, bb]"
3,C,"C1,C2,C3,C4","ca,cb,cc","[C1, C2, C3, C4]","[ca, cb, cc]"


In [77]:
df_alias = df.alias.explode()
df_alias

id
1    aa
1    ab
1    ac
2    ba
2    bb
3    ca
3    cb
3    cc
Name: alias, dtype: str

In [78]:
df_alias = df_alias.to_frame()
df_alias

,alias
id,
1,aa
1,ab
1,ac
2,ba
2,bb
3,ca
3,cb
3,cc


In [79]:
df_alias["gene_id"] = df_alias.index
df_alias

,alias,gene_id
id,,
1,aa,1
1,ab,1
1,ac,1
2,ba,2
2,bb,2
3,ca,3
3,cb,3
3,cc,3


In [80]:
df_alias.reset_index(drop=True, inplace=True)
df_alias

,alias,gene_id
0,aa,1
1,ab,1
2,ac,1
3,ba,2
4,bb,2
5,ca,3
6,cb,3
7,cc,3


In [81]:
df_alias.index += 1
df_alias

,alias,gene_id
1,aa,1
2,ab,1
3,ac,1
4,ba,2
5,bb,2
6,ca,3
7,cb,3
8,cc,3


In [82]:
df_alias.index.rename("id", inplace=True)
df_alias

,alias,gene_id
id,,
1,aa,1
2,ab,1
3,ac,1
4,ba,2
5,bb,2
6,ca,3
7,cb,3
8,cc,3


In [83]:
df_alias.to_sql("alias", conn, index_label="id", if_exists="replace")

8

In [84]:
'''SELECT column_name1,column_name2,..
FROM table_name1
INNER JOIN 
table_name2
ON condition_1
INNER JOIN 
table_name3
ON condition_2
INNER JOIN 
table_name4
ON condition_3'''

'SELECT column_name1,column_name2,..\nFROM table_name1\nINNER JOIN \ntable_name2\nON condition_1\nINNER JOIN \ntable_name3\nON condition_2\nINNER JOIN \ntable_name4\nON condition_3'

In [85]:
sql = """
    SELECT g.symbol, s.synonym, a.alias
    FROM gene AS g
    INNER JOIN
    synonym AS s
    ON (g.id = s.gene_id)
    INNER JOIN
    alias AS a
    ON (g.id = a.gene_id)
"""

pd.read_sql(sql, conn)

,symbol,synonym,alias
0,A,A1,aa
1,A,A1,ab
2,A,A1,ac
3,A,A2,aa
4,A,A2,ab
5,A,A2,ac
6,A,A3,aa
7,A,A3,ab
8,A,A3,ac
9,B,B1,ba
